# Task 15b (server) — alternative-only supervision and contrastive calibration

Server twin of `research_tasks_15b_objective_and_contrastive.ipynb` (Colab). These
are the arms the paper contributes: **alternative-only** concept supervision
(`--exclude-target`, uniform over the alternatives) and the same with
**hard-negative contrastive** calibration.

**Run it in the SAME directory as the Task 15 pass for this model.** Every arm
here is compared against that pass's NTP, Zhang and randomized adapters, and it
reads them through `run_manifests/task15.json` and `results/` under the same
`CONCEPT_BASE`. Point it somewhere else and there is nothing to compare against.
Nothing is re-extracted: the concept data is already on disk.

```bash
cd <the Task 15 directory for this model>     # e.g. ~/fos_retrieval/task15
export CONCEPT_BASE=$PWD
papermill reproducibilty_15b.ipynb executed_15b.ipynb -k concept --log-output 2>&1 | tee -a 15b.log
```

## What it trains

With `SELECTED_ALPHA` set in the control cell (the replication path), `RUN_SCREEN`
trains eight arms at seed 42:

| arms | why |
|---|---|
| uniform α ∈ {0.25, 0.5, 1.0} | the objective, at three weights |
| uniform α = 0.75 | frontier control: a contrastive arm must beat the uniform *curve*, not just the same-α point |
| inclusive uniform α | identical loss with the observed target back in the set — isolates the exclusion |
| contrastive β ∈ {0.25, 0.5, 1.0} | hard negatives at the locked α |

The α = 0.75 point exists because contrastive arms land between α = 0.5 and
α = 1.0 on both axes, so two uniform points cannot say whether they sit above the
uniform curve or merely on it. Any contrastive claim is tested against a HIGHER-α
uniform arm, never only against the same α.

## Costs, measured on an A40 with Qwen3-1.7B-Base

Roughly 30 min per training arm, so ~4 h for the eight. Evaluation then scores
these plus the Task 15 comparators — about 17 checkpoints — at ~4.5 min for
SWORDS and ~5 min for STS each, so ~4-5 h. Call it 9-10 h end to end, and every
stage resumes.

## 1. Which GPUs are free

Run this first, then name one in the control cell below. Getting it wrong means restarting the kernel, not just re-running a cell.

In [ ]:
# Which cards are free.  Pick one with spare memory and no other process, then
# name it in the control cell below -- BEFORE anything here touches CUDA.
!nvidia-smi --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv

## 2. The one cell to edit

GPU, model, HF token and the stage gates. Everything after this reads them from the environment, including every child process.

In [ ]:
# ==== THE ONLY CELL YOU EDIT.  Set these, then Run All. ======================
import os

GPU_ID   = "1"                      # from the table above
MODEL    = "Qwen/Qwen3-1.7B-Base"   # must be a model whose Task 15 pass already
                                    # finished: the arms here are scored against its
                                    # NTP, Zhang and randomized adapters.
HF_TOKEN = ""                       # gated models only (Llama).  Qwen3 is open.
                                    # If you do paste one, CLEAR IT BEFORE SAVING.

# Alpha was selected on Llama-3.2-1B's C4 validation and beta on SWORDS dev.
# Setting them here TRANSFERS those values instead of tuning again on this model.
# That is the intended replication: re-tuning per model would read this model's
# data for selection and make the second family a second tuning round rather
# than a test.  Say so in the write-up.  Leave both None to run the alpha screen
# and read the decision cell's gate, the way the first model did it.
SELECTED_ALPHA = 0.5
SELECTED_BETA  = 1.0

RUN_DATA    = True    # mine WordNet hard negatives and write the negatives view (minutes)
RUN_SCREEN  = True    # alpha sweep {0.25,0.5,1.0}; then, because SELECTED_ALPHA is set,
                      # the inclusive-set ablation, the beta screen {0.25,0.5,1.0} and the
                      # alpha=0.75 frontier point -- 8 arms, ~30 min each on an A40
RUN_CONFIRM = False   # the locked arms across SEEDS.  Needs RUN_MULTISEED for a real
                      # three-seed confirmation; on its own it loops over [42], which the
                      # screen has already trained.
RUN_MULTISEED = False # add seeds 123 and 2024 to SEEDS
RUN_HYBRID  = False   # Zhang + alternative-only auxiliary term (5 arms) and the two
                      # negative-quality controls, seed 42.  Off until the 1B gate on
                      # Colab has picked one; then set SELECTED_HYBRID and run only that.
SELECTED_HYBRID = None  # e.g. "within_kl:0.5" -- transferred from the 1B gate, never tuned here
RUN_NEGATIVE_CONTROLS = False  # clean/fragments negative diagnostic; not method selection
RUN_VERIFIED = False  # six arms from synonyms_train_verified.jsonl, one slot objective each,
                      # scored in their own directory against Task 15's NTP and Zhang
VERIFIED_SMOKE_STEPS = 0     # >0: ~that many steps per arm, print first-step magnitudes, train nothing else
VERIFIED_LAMBDA = 1.0        # weight on the slot objective for pool/rank/list arms -- declared, not tuned
VERIFIED_GAMMA  = 0.0625     # the continuity arm's gamma from the dev frontier: Qwen .0625, Llama .125
RUN_EVAL    = True    # score these arms AND the Task 15 comparators: SWORDS with paired
                      # bootstrap intervals, STS, perplexity, concept sets, bm-semlex
SPACY_GPU   = True    # only matters if this model still needs extraction

# Everything below reads these from the environment, which is also how they reach
# each child process.  Set any value to None to defer to a variable exported in
# the shell instead -- that is what the nbconvert commands in the cell above use.
for _name, _value in {"GPU_ID": GPU_ID, "CONCEPT_MODEL": MODEL, "HF_TOKEN": HF_TOKEN or None,
                      "SELECTED_ALPHA": SELECTED_ALPHA, "SELECTED_BETA": SELECTED_BETA,
                      "RUN_DATA": RUN_DATA, "RUN_SCREEN": RUN_SCREEN,
                      "RUN_HYBRID": RUN_HYBRID, "SELECTED_HYBRID": SELECTED_HYBRID,
                      "RUN_NEGATIVE_CONTROLS": RUN_NEGATIVE_CONTROLS,
                      "RUN_VERIFIED": RUN_VERIFIED, "VERIFIED_SMOKE_STEPS": VERIFIED_SMOKE_STEPS,
                      "VERIFIED_LAMBDA": VERIFIED_LAMBDA, "VERIFIED_GAMMA": VERIFIED_GAMMA,
                      "RUN_CONFIRM": RUN_CONFIRM, "RUN_EVAL": RUN_EVAL,
                      "RUN_MULTISEED": RUN_MULTISEED, "SPACY_GPU": SPACY_GPU}.items():
    if _value is not None:
        os.environ[_name] = ("1" if _value else "0") if isinstance(_value, bool) else str(_value)

# CUDA_VISIBLE_DEVICES has to be set before any torch import initialises the
# driver, and the install cell below imports torch to decide about torchao.  The
# setup cell sets it too, from GPU_ID -- but by then the choice can already be
# locked in, and the run would quietly land on card 0.
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.setdefault("GPU_ID", "1")
print("GPU", os.environ["CUDA_VISIBLE_DEVICES"],
      "|", os.environ.get("CONCEPT_MODEL", "(default)"),
      "| alpha", os.environ.get("SELECTED_ALPHA", "(screen)"),
      "| beta", os.environ.get("SELECTED_BETA", "(screen)"),
      "| stages:", " ".join(stage for stage in ("RUN_DATA", "RUN_SCREEN", "RUN_HYBRID", "RUN_VERIFIED",
                                                "RUN_CONFIRM", "RUN_EVAL")
                            if os.environ.get(stage) == "1"))

## 3. Dependencies

Once per environment. The `transformers` pin matters: upstream's extractor reuses a prefix KV cache through an API removed in v5.

In [ ]:
# Install once per environment.  transformers is pinned LAST and BELOW 4.58 on
# purpose: upstream's extractor reuses a prefix KV cache through
# DynamicCache.from_legacy_cache, which transformers removed in v5.
%pip install -q accelerate peft bitsandbytes datasets spacy "mteb>=1.12" nltk scipy scikit-learn seaborn pandas pytest wandb
%pip install -q "transformers>=4.51,<4.58"
# torchao is a Colab-era fix: peft raises on torchao < 0.16 from inside
# PeftModel.from_pretrained.  But a CURRENT torchao evaluates torch.int1 at
# import, which torch < 2.6 does not define, and transformers imports torchao
# unconditionally whenever it is installed -- so on an older torch a mismatched
# torchao makes EVERY model unloadable, with the traceback pointing at the model
# class rather than at torchao.  Install it only where it helps; remove it
# otherwise, which is safe because peft only needs it when it is present.
import subprocess, sys, torch
_version = tuple(int(part) for part in torch.__version__.split(".")[:2])
if _version >= (2, 6):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao>=0.16"], check=False)
    print("torchao: installed for torch", torch.__version__)
else:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=False)
    print("torchao: removed, torch", torch.__version__, "predates torch.int1")
# cupy backs spacy.require_gpu(); without it thinc raises and SPACY_GPU must be
# set False.  thinc reads cupy's presence at import, so install before spacy loads.
%pip install -q cupy-cuda12x
!python -m spacy download en_core_web_sm
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

In [ ]:
import os
# Set in the control cell above; re-applied here so this cell stands alone when
# it is re-run on its own, and so the headless path (GPU_ID=2 jupyter nbconvert
# --execute ...) works with the control cell's GPU_ID set to None.
GPU_ID = os.environ.get("GPU_ID", "1")
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

from pathlib import Path
import datetime, hashlib, json, re, shutil, subprocess, sys, torch

# Every clone, dataset, checkpoint and result lives under BASE, so the whole
# experiment is one directory to archive or copy off the server.
def _notebook_base():
    """Directory the notebook itself lives in, so `outputs/` lands beside it.

    CONCEPT_BASE wins when set.  Otherwise prefer JPY_SESSION_NAME, which Jupyter
    sets to the notebook's own path: the working directory is the notebook's
    directory only when the kernel happened to start there, so `jupyter nbconvert
    --execute` invoked from anywhere else would scatter a second outputs/ tree
    next to wherever it was run from.
    """
    override = os.environ.get("CONCEPT_BASE")
    if override:
        return Path(override)
    session = os.environ.get("JPY_SESSION_NAME", "")
    if session.endswith(".ipynb") and Path(session).parent.is_dir():
        return Path(session).parent
    return Path.cwd()

BASE = _notebook_base().resolve()
WORK = BASE / "concept_aware"
MAIN = WORK / "concept-aware-training"     # our repo: patch, evaluators, scripts
EXT = WORK / "learning-concepts"           # upstream, pinned
DATA = WORK / "data"                       # CONCEPT_DATA_ROOT
RUNS = WORK / "runs"                       # adapters (small at r=4, kept)
OUTPUTS = BASE / "outputs"                 # everything you download for analysis

# ONE model per pass.  Every artefact below is keyed on the model tag and the
# resume logic reads finished work back by path, so the model is selected in the
# control cell rather than looped over here -- which is also what lets two models
# share the machine without sharing a GPU.
BASE_MODEL = os.environ.get("CONCEPT_MODEL", "Qwen/Qwen3-1.7B-Base")
# Every per-model artifact is keyed on this tag.  Two models must never share a
# path: adapters resume by path, so a collision hands one model's weights to
# another and the run still looks like it succeeded.  conceptlib.paths derives
# the same tag the same way, so the data directories line up with these.
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
# combined.jsonl depends only on the tokenizer, so a model sharing a vocabulary
# with one already extracted produces a byte-identical file.  Naming the donor
# skips the spaCy POS pass; the vocabularies are compared before the copy and the
# run aborts if they differ.  Each pair shares a tokenizer WITHIN its family and
# never across families, which is why this is a table and not a size heuristic.
VOCAB_DONOR = {"Qwen/Qwen2.5-3B": "Qwen/Qwen2.5-1.5B",
               "Qwen/Qwen3-4B-Base": "Qwen/Qwen3-1.7B-Base",
               "Qwen/Qwen3-4B": "Qwen/Qwen3-1.7B",
               "meta-llama/Llama-3.2-3B": "meta-llama/Llama-3.2-1B"}
REUSE_CONTENT_WORDS_FROM = os.environ.get("CONCEPT_REUSE_FROM") or VOCAB_DONOR.get(BASE_MODEL)

# outputs/<model tag>/{data_audit.json,results,logs,run_manifests}, plus one
# shared directory for the two model-independent reports.  Unlike the Colab
# notebook there is no legacy unsuffixed path to preserve here, so EVERY model
# gets its own directory -- including the first one.
MODEL_OUT = OUTPUTS / MODEL_TAG
RESULT_DIR = MODEL_OUT / "results"
LOG_DIR = MODEL_OUT / "logs"
SHARED = OUTPUTS / "shared"
DATA_AUDIT = MODEL_OUT / "data_audit.json"
PRIMARY_SEED = 42
# One seed screens the pipeline and shows the direction of every effect, but it
# CANNOT support a claim: the pre-registered rule needs all three to agree in
# sign.  Flip to True for the reportable run; finished arms are skipped.
def _flag(name, default):
    """Read a run flag from the environment, defaulting to the value here.

    The control cell writes the flags into the environment rather than binding
    them here, so the same file also runs headless -- `CONCEPT_MODEL=...
    RUN_DATA=1 jupyter nbconvert --execute` drives one stage under tmux, which
    survives a dropped VPN, and the notebook stays reproducible either way.
    """
    return os.environ.get(name, str(default)).strip().lower() in ("1", "true", "yes", "on")

# Requires cupy (installed below).  Verified output-identical to CPU spaCy.
SPACY_GPU = _flag("SPACY_GPU", True)
RUN_MULTISEED = _flag("RUN_MULTISEED", False)
SEEDS = [PRIMARY_SEED] + ([123, 2024] if RUN_MULTISEED else [])
UPSTREAM_COMMIT = "b1d414143d11c8ed988b4cccbb06626cc8272bbe"

# Capture an existing `huggingface-cli login` BEFORE redirecting HF_HOME.  The
# token lives under the DEFAULT HF_HOME, so once we move HF_HOME into the project
# directory a freshly spawned child finds no token and gated downloads 401 --
# even though the parent, which imported huggingface_hub earlier, looks fine.
# Exporting HF_TOKEN makes auth explicit and inherited by every subprocess.  A
# token pasted into the control cell is already in the environment and wins here.
_cli_token = Path.home() / ".cache" / "huggingface" / "token"
if not os.environ.get("HF_TOKEN") and _cli_token.is_file():
    os.environ["HF_TOKEN"] = _cli_token.read_text().strip()
os.environ["HF_HOME"] = str(WORK / "hf_cache")

# Defaults for the Qwen3-1.7B-Base server pass; the control cell sets all of them, so
# these apply only when a flag is left None there or this cell is re-run alone.
# The Colab notebook keeps them False because a stray Run All there costs money
# and a session slot.  Here the whole point is an unattended pass, and every
# stage is resumable, so an accidental start costs the shard in flight.
RUN_DATA = _flag("RUN_DATA", True)
RUN_SMOKE = _flag("RUN_SMOKE", True)
RUN_SCREEN = _flag("RUN_SCREEN", True)
# Three seeds are a Colab job for the headline arms only; Qwen is a second-family
# replication at seed 42, so this stays off.
RUN_CONFIRM = _flag("RUN_CONFIRM", False)
# Task 15b only, and only after the 1B gate has chosen an arm.
RUN_HYBRID = _flag("RUN_HYBRID", False)
RUN_NEGATIVE_CONTROLS = _flag("RUN_NEGATIVE_CONTROLS", False)
RUN_VERIFIED = _flag("RUN_VERIFIED", False)
VERIFIED_SMOKE_STEPS = int(float(os.environ.get("VERIFIED_SMOKE_STEPS", "0")))
RUN_EVAL = _flag("RUN_EVAL", True)
# Skip any run whose artefacts already exist.  A killed job resumes from here.
RESUME_FINISHED_RUNS = True

# Extraction precision.  Upstream inherits use_4bit=True from TrainingConfig,
# where it exists for QLoRA TRAINING.  Extraction runs ~94 small forwards per
# sequence; measured on an L4, bf16 was ~25% faster and avoids quantisation
# noise in the top-100 pool and the 0.75 cosine threshold the method depends on.
EXTRACT_4BIT = False
# Must match the Colab notebook: the two variants feed one study, and a model
# extracted here on 10,000 sequences could not be compared with one extracted
# there on 4,000.  Zhang et al. Fig. 8 reports STS unchanged at a quarter of the
# data; 4,000 keeps the 80/10/10 ratio.  Raise BOTH to 10000 for the strict
# reproduction.  merge_synonym_parts hard-fails unless the split sizes sum to the
# rows the shards actually cover.
EXTRACT_SEQUENCES = 4000
# Sequences per extraction shard, and so exactly what a killed job costs: a shard
# is only recorded as finished once it completes.
EXTRACT_SHARD = 500
SPLIT_TRAIN = int(EXTRACT_SEQUENCES * 0.8)
SPLIT_VAL = SPLIT_TEST = int(EXTRACT_SEQUENCES * 0.1)

_BAR = re.compile(r"\b(\d+)/(\d+)\s*\[")     # bounded tqdm: "  200/1000 ["
_BAR_OPEN = re.compile(r"\b(\d+)it\s*\[")     # unbounded tqdm: "  3200it [00:49"
PROGRESS_EVERY = 100

def run(argv, cwd=None, env=None):
    """Run a child process, streaming its output and keeping the tail on failure."""
    argv = list(map(str, argv))
    print("+", " ".join(argv), flush=True)
    merged = os.environ.copy()
    merged.update({"CONCEPT_DATA_ROOT": str(DATA),
                   "CONCEPT_CHECKPOINT_ROOT": str(RUNS),
                   "CONCEPT_RESULTS_ROOT": str(OUTPUTS),
                   # The POS filter is ~90% of extraction; on GPU it ran 8.5x
                   # faster on an A40 (42.6 -> 5.0 s/seq) with 12400/12400 POS
                   # tags identical to CPU.  SPACY_GPU gates it because it needs
                   # cupy, and because the Colab-produced Llama data did not use it.
                   "CONCEPT_SPACY_GPU": "1" if SPACY_GPU else "0",
                   # This machine has no locale set, so Python defaults to ASCII
                   # and both the child and its captured output die on C4's
                   # non-ASCII text, partway through a long run.
                   "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"})
    if env: merged.update(env)
    process = subprocess.Popen(argv, cwd=cwd, env=merged, text=True, bufsize=1,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in process.stdout:
        line = line.replace("\r", "")
        tail.append(line)
        del tail[:-40]
        hit = _BAR.search(line)
        if hit:
            done, total = int(hit.group(1)), int(hit.group(2))
            if done % PROGRESS_EVERY and done != total:
                continue
        else:
            # Dataset loading has no total ("3200it [00:49"), so there is no final
            # count to anchor on.  Thin it ten times harder; it is pure noise and
            # a full sweep would otherwise emit tens of thousands of lines.
            loose = _BAR_OPEN.search(line)
            if loose and int(loose.group(1)) % (PROGRESS_EVERY * 10):
                continue
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(
            f"command failed with exit code {code}\n  {' '.join(argv)}\n"
            f"--- last {len(tail)} lines of its output ---\n{''.join(tail)}")

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def sync_small_artifacts(source, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    for path in Path(source).rglob("*"):
        if path.is_file() and path.suffix.lower() in {".json", ".jsonl", ".csv", ".png", ".log"}:
            target = destination / path.relative_to(source)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)

def audit_outputs():
    """Keep OUTPUTS downloadable: reports only, no multi-GB weights."""
    forbidden = {"pytorch_model.bin", "model.safetensors", "optimizer.pt",
                 "scheduler.pt", "scaler.pt", "rng_state.pth"}
    found = [str(p) for p in OUTPUTS.rglob("*")
             if p.name in forbidden or p.name.startswith("checkpoint-")]
    assert not found, f"full-weight/optimizer artifacts reached OUTPUTS: {found}"

# The server filesystem persists, so the Colab Drive round-trip is unnecessary.
# These keep the same names and call sites as the Colab notebook, which is what
# lets the two share every experiment cell below.
def cache_dataset(leaf): pass

def restore_dataset(leaf):
    return (Path(leaf) / "synonyms_train.jsonl").is_file()

def cache_adapter(path): return Path(path)

def restore_adapter(path):
    return (Path(path) / "adapter_config.json").is_file()

# Inside THIS model's output directory.  A manifest shared between models would
# name another model's adapters, and restore_all would load them without
# complaint -- the resume path has no way to tell whose weights it just read.
RUN_MANIFEST = MODEL_OUT / "run_manifests"

def save_runs(runs, name):
    RUN_MANIFEST.mkdir(parents=True, exist_ok=True)
    (RUN_MANIFEST / f"{name}.json").write_text(
        json.dumps({k: str(v) for k, v in runs.items()}, indent=2))

def load_runs(name):
    path = RUN_MANIFEST / f"{name}.json"
    return {} if not path.is_file() else {k: Path(v) for k, v in json.loads(path.read_text()).items()}

def restore_all(runs):
    live = {}
    for label, path in runs.items():
        if restore_adapter(path):
            live[label] = Path(path)
        else:
            print("missing adapter, dropping from this pass:", label)
    return live

def eval_done(marker):
    return Path(marker).is_file() and RESUME_FINISHED_RUNS

def eval_covered(path, checkpoints):
    """True when `path` already scores every checkpoint of THIS pass.

    Coverage, not mere existence: adding a seed grows `checkpoints`, the old file
    stops covering it, and the evaluator reruns.  A plain existence check would
    report the previous pass's table as if it were this one's.
    """
    if not (RESUME_FINISHED_RUNS and Path(path).is_file()):
        return False
    try:
        rows = json.loads(Path(path).read_text())
    except (json.JSONDecodeError, OSError):
        return False              # truncated by a killed job mid-write; redo it
    if not isinstance(rows, list):
        return False
    scored = {str(row.get("checkpoint")) for row in rows if isinstance(row, dict)}
    return set(map(str, checkpoints)) <= scored

def sts_covered(path):
    """True when one STS pass already wrote its nine task rows to `path`."""
    if not (RESUME_FINISHED_RUNS and Path(path).is_file()):
        return False
    with open(path, encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip()) >= 10   # header + 9 tasks

def guarded(path, checkpoints, argv, cwd, what):
    """Run one evaluator unless its output already covers every checkpoint."""
    if eval_covered(path, checkpoints):
        print(f"resume: {what} already covers {len(checkpoints)} checkpoints, skipping")
        return
    run(argv, cwd=cwd)

def assert_under_base(path):
    resolved = Path(path).resolve()
    assert str(resolved).startswith(str(BASE)), f"{resolved} escapes {BASE}"

for directory in (WORK, DATA, RUNS, OUTPUTS, MODEL_OUT, RESULT_DIR, LOG_DIR, SHARED):
    directory.mkdir(parents=True, exist_ok=True)
print("BASE      ", BASE)
print("MODEL     ", BASE_MODEL, "->", MODEL_TAG)
print("REUSE FROM", REUSE_CONTENT_WORDS_FROM or "(nothing: full extraction)")
print("OUTPUTS   ", MODEL_OUT)
print("GPU       ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
# Both clones are required: MAIN carries the patch, the evaluators and the
# summary script; EXT is the pinned upstream the patch applies to.
if not MAIN.exists():
    run(["git", "clone", "https://github.com/SharvaGogawale1/concept-aware-training.git", MAIN])
else:
    run(["git", "-C", str(MAIN), "pull", "--ff-only"])
if not EXT.exists():
    run(["git", "clone", "https://github.com/christine-zhang1/learning-concepts.git", EXT])
# Reset to the pinned commit and wipe every patch artefact before re-applying.
# Testing "does it apply, else does it reverse-apply" only worked while the patch
# never changed: once MAIN pulls a newer one, the old patch is applied, neither
# direction matches, and the run dies on an assertion.  Resetting is idempotent
# and always ends in the same state.  EXT holds upstream code only -- the corpus
# lives in DATA, outside it -- so clean -fd is safe.
run(["git", "-C", str(EXT), "reset", "--hard", UPSTREAM_COMMIT])
run(["git", "-C", str(EXT), "clean", "-fdq"])

patch_file = MAIN / "external" / "learning-concepts.patch"
assert patch_file.exists(), f"{patch_file} missing; push it before running here."
run(["git", "apply", str(patch_file)], cwd=EXT)
print("patch applied onto", UPSTREAM_COMMIT[:7])

run([sys.executable, "-m", "pip", "install", "-q", "-e", str(EXT), "--no-deps"])

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

run([sys.executable, MAIN / "builddataset/verify_task14_data.py",
     "--repo_root", MAIN, "--download_missing",
     "--report_json", SHARED / "external_benchmark_integrity.json"], cwd=MAIN)
(SHARED / "environment_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))

# Prove the model is reachable FROM A SUBPROCESS, which is where every download
# actually happens.  The parent can hold credentials a freshly spawned child does
# not inherit, and the failure then surfaces 20 minutes later as a bare 401 on
# config.json whose traceback says nothing about authentication.
# The token is optional because gating is: Qwen2.5 is openly licensed, Llama-3.2
# is gated.  Demanding a token unconditionally would block a Qwen-only run for no
# reason, so let the reachability probe be the thing that decides.
from huggingface_hub import login, whoami
if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("Hugging Face:", whoami()["name"])
else:
    print("No HF_TOKEN set; continuing on the assumption this model is ungated.")
_probe = subprocess.run(
    [sys.executable, "-c",
     "import sys; from transformers import AutoConfig;"
     "AutoConfig.from_pretrained(sys.argv[1]);"
     "print('model repo reachable from a subprocess')", BASE_MODEL],
    env={**os.environ}, capture_output=True, text=True)
assert _probe.returncode == 0, (
    f"{BASE_MODEL} is not reachable from a child process.  If it is a gated repo, "
    "run `huggingface-cli login` on this machine or export HF_TOKEN before "
    f"starting Jupyter, then re-run this cell.\n{_probe.stderr[-2000:]}")
print(_probe.stdout.strip())

In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
# The server filesystem persists, so this is a cheap existence check.
restore_dataset(LEAF)

def adapter_path(method, seed, value):
    path = RUNS / MODEL_TAG / method / f"seed_{seed}" / str(value)
    assert_under_base(path)
    return path

def finished(path):
    """A run counts as finished when its adapter is on disk."""
    if (Path(path) / "adapter_config.json").is_file():
        return True
    return restore_adapter(path)

def train_flat(method, seed, concept_weight, *, objective="set_marginal",
               slot_ntp_weight=None, contrast_beta=0.0, exclude_target=False,
               randomized=False, data_augmentation=False, epochs=5, train_file=None,
               max_samples=None, batch=8, accum=2, alt_aux="none", alt_aux_weight=0.0):
    # The aux suffix is added only when the term is on, so every adapter trained
    # before it existed keeps its path and still resumes.
    aux_tag = "" if alt_aux == "none" else f"_aux_{alt_aux}_{alt_aux_weight}"
    out = adapter_path(method, seed, f"lambda_{concept_weight}_beta_{contrast_beta}{aux_tag}")
    args = [sys.executable, "train.py", "--model-name", BASE_MODEL, "--dataset", "c4",
            "--dataset-type", "embedding", "--concept-loss-weight", concept_weight,
            "--concept-objective", objective, "--contrast-beta", contrast_beta,
            "--seed", seed, "--num-train-epochs", epochs, "--output-dir", out,
            "--save-strategy", "no", "--report-to", "none",
            "--per-device-train-batch-size", batch,
            "--gradient-accumulation-steps", accum]
    if slot_ntp_weight is not None: args += ["--slot-ntp-weight", slot_ntp_weight]
    if alt_aux != "none": args += ["--alt-aux", alt_aux, "--alt-aux-weight", alt_aux_weight]
    # Drops the observed target from the concept set, so the loss cannot be paid
    # with the mass NTP already put there.  Pair it with slot_ntp_weight=1.0 or
    # the observed target is pushed down.  adapter_path() does not encode this
    # flag, so the METHOD name must differ from the target-inclusive arm's.
    if exclude_target: args += ["--exclude-target"]
    if randomized: args += ["--randomized-synonyms"]
    if data_augmentation: args += ["--use-data-augmentation"]
    if train_file: args += ["--train-file", train_file]
    if max_samples: args += ["--max-train-samples", max_samples]
    # Released effective batch is 8 x 2 = 16; it is recorded in every config.
    if RESUME_FINISHED_RUNS and finished(out):
        print("resume: already trained, skipping", out)
        return out
    run(args, cwd=EXT)
    cache_adapter(out)
    return out

## Build conservative negatives, and a 50-row sample to read by hand

Candidates must occur in the model’s top-100 next-token pool, match POS, lie outside the alternative set, share no WordNet synset with any alternative, not be a morphological variant, and fall below the contextual-similarity ceiling. Coverage and every rejection reason are reported. The 50-row sample is an error analysis, not an annotation project: read it before trusting any contrastive number.


In [ ]:
MODEL_TAG = BASE_MODEL.split("/")[-1].lower()
LEAF = DATA / "c4" / MODEL_TAG / "embedding"
NEG_TRAIN = LEAF / "synonyms_train_conservative_negatives.jsonl"
SCREEN_DIR = MODEL_OUT / "task15b_screen"
SCREEN_DIR.mkdir(parents=True, exist_ok=True)
if RUN_DATA:
    run([sys.executable, "data/build_contrastive_negatives.py",
         "--source", LEAF / "synonyms_train.jsonl",
         "--topk", DATA / "c4" / MODEL_TAG / "prompting" / "topk_*.jsonl",
         "--output", NEG_TRAIN,
         # Per model: a shared name would let the 3B pass overwrite the 1B report.
         "--report", MODEL_OUT / "contrastive_negative_report.json",
         "--max-cosine", "0.35", "--max-negatives", "20"], cwd=EXT)
    cache_dataset(LEAF)
    # Stratified by POS and alternative-set size so the sample cannot be all easy
    # nouns with large sets.  Fixed seed: the same 50 rows every time it is rerun.
    import csv, random
    buckets = {}
    with NEG_TRAIN.open(encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            for target in row.get("content_word_responses", []):
                if not target.get("negatives"):
                    continue
                key = (target.get("pos") or "?", "small" if len(target.get("synonyms", [])) <= 2 else "large")
                buckets.setdefault(key, []).append({
                    "pos": key[0], "set_size": key[1], "context": row["input_sequence"],
                    "observed_target": target["word"],
                    "alternatives": " | ".join(target.get("synonyms", [])),
                    "negatives": " | ".join(target["negatives"]),
                    "negative_scores": " | ".join(f"{s:.3f}" for s in target.get("negative_scores", []))})
    rng = random.Random(42)
    picked, per_bucket = [], max(1, 50 // max(1, len(buckets)))
    for key in sorted(buckets):
        picked += rng.sample(buckets[key], min(per_bucket, len(buckets[key])))
    remainder = [item for key in sorted(buckets) for item in buckets[key] if item not in picked]
    picked += rng.sample(remainder, min(50 - len(picked), len(remainder)))
    sample_path = SCREEN_DIR / "negative_sample_50.csv"
    with sample_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(picked[0].keys()) if picked else ["context"])
        writer.writeheader(); writer.writerows(picked)
    print(f"wrote {len(picked)} rows across {len(buckets)} strata to", sample_path)

## Screen on seed 42

$\alpha\in\{.25,.5,1\}$ controls concept pressure on the alternative-only set; the concept-slot NTP weight is 1 in every arm, so the observed target is always trained. Uniform (mean-log) losses are roughly an order of magnitude larger per slot than set-marginal at the same weight, which is why this grid starts lower than Zhang's.

$\alpha$ is selected on **C4 validation** (lowest alternative NLL subject to the global-NLL, observed-target-NLL and collapse gates printed by the decision cell). Only then is one target-inclusive ablation trained at that $\alpha$, and $\beta\in\{.25,.5,1\}$ screened on **SWORDS dev**. SWORDS test is never read during tuning.


In [ ]:
OBJECTIVE_RUNS = load_runs("task15b")
if RUN_SCREEN:
    for alpha in [0.25, 0.5, 1.0]:
        OBJECTIVE_RUNS[f"alternative_uniform_alpha{alpha}_seed42"] = train_flat(
            "alternative_uniform", 42, alpha, objective="uniform", slot_ntp_weight=1.0,
            exclude_target=True)

# Set this ONLY from the alpha gate printed by the decision cell below -- or,
# when REPLICATING on another model, to the value that gate already selected
# elsewhere, passed in as SELECTED_ALPHA=0.5 rather than edited in here.  A
# transferred value is a deliberate choice and belongs in the write-up: it means
# alpha was not tuned on this model, which is the stronger claim and also the
# only honest one once the test set has been read for the first model.
SELECTED_ALPHA = float(os.environ["SELECTED_ALPHA"]) if os.environ.get("SELECTED_ALPHA") else None
if RUN_SCREEN and SELECTED_ALPHA is not None:
    # The one ablation that isolates the exclusion: identical loss, observed
    # target back inside the concept set.  Not a headline method.
    OBJECTIVE_RUNS[f"inclusive_uniform_alpha{SELECTED_ALPHA}_seed42"] = train_flat(
        "inclusive_uniform", 42, SELECTED_ALPHA, objective="uniform", slot_ntp_weight=1.0)
    for beta in [0.25, 0.5, 1.0]:
        OBJECTIVE_RUNS[f"contrast_alpha{SELECTED_ALPHA}_beta{beta}_seed42"] = train_flat(
            "alternative_contrastive", 42, SELECTED_ALPHA, objective="uniform",
            slot_ntp_weight=1.0, exclude_target=True, contrast_beta=beta, train_file=NEG_TRAIN)
    # Frontier point.  Contrastive arms land between alpha=.5 and alpha=1 on BOTH
    # axes, so two uniform points cannot say whether they sit above the alpha
    # curve or merely on it.  Not an alpha-selection candidate: alpha was chosen
    # on validation from {.25,.5,1} before this ran, and this is scored after.
    OBJECTIVE_RUNS["alternative_uniform_alpha0.75_seed42"] = train_flat(
        "alternative_uniform", 42, 0.75, objective="uniform", slot_ntp_weight=1.0,
        exclude_target=True)

## Calibrated concept marginalization, and what the negatives were really doing

**Why.** Decompose the SWORDS ranking gain. GAP rises either by pushing rejected candidates down or by pulling accepted ones up, and set-marginal training does only the first. Across Llama-1B (3 seeds), Llama-3B and Qwen3-1.7B it lowers rejected-mass share every time (−.014, −.014, −.021) and never lowers NLL on human-accepted alternatives (−.012 n.s., −.046 n.s., **+.088 worse**). Alternative-only supervision moves precisely that second axis, by the same amount in both families (−.50 nats on Llama, −.485 [−.568,−.403] on Qwen). So the two objectives are not rivals; one contains the other. With $q = p / P(S)$ the model's own distribution inside the set,

$$-\tfrac{1}{n}\sum_{c\in S}\log p_c \;=\; \underbrace{-\log P(S)}_{\text{Zhang}} \;+\; \underbrace{\mathrm{KL}(u\,\|\,q)}_{\text{within-set}} \;+\; \log n ,$$

and $\nabla_z[-\log P(S)] = p - q\,\mathbb{1}_S$: the set marginal is self-distillation toward the model's current within-set distribution, so nothing in it says *which* members deserve mass. A semantically randomized control is a second, weaker probe of the same point: on Llama it reproduces most of the ranking gain (+.008 of Zhang's +.011), on Qwen it reproduces none (−.001). That comparison is therefore **model-dependent and reported as such**; the decomposition above is what replicates. These arms keep Zhang's term exactly as released (target-inclusive, $\lambda=1$, 0.6 cutoff) and add one term on the alternatives only:

- `uniform` — $-\tfrac1n\sum\log p_a$: raises the alternatives and spreads them.
- `within_kl` — $\mathrm{KL}(u\|q_A)$ alone. Its logit gradient is zero outside the alternatives and sums to zero inside them, so it moves mass *between* alternatives and leaves the observed word and the set's total mass to Zhang's term. If SWORDS moves under this arm, the gain is within-set calibration; if only `uniform` moves it, the gain is mass, and the decomposition says so.

**Gate, fixed before any of these is scored (SWORDS dev only):** STS $\ge .5469$; GAP and AUROC above Zhang with paired intervals excluding zero; above randomized $\lambda=.25$ on GAP and AUROC; global NLL $\le$ NTP $+.20$; observed-target NLL $\le$ NTP $+.10$. The smallest weight that passes is the method. If none passes, the result is the STS-vs-GAP frontier these arms trace, reported as a trade-off.

**The negatives.** A hand read of `negative_sample_50.csv` (2026-09-18): the contrastive loss only fires where a slot has an alternative (31/50), and in 23 of those 31 every negative is a letter or a word-prefix token; 2 of 31 have a semantically meaningful negative. The 0.35 similarity ceiling rejects every real word in a slot with rich alternatives, so only non-words survive, and WordNet lists them. Two controls settle what the published +.005 GAP was: `clean` (complete words in their dominant POS, plus antonyms of the observed word) and `fragments` (only what `clean` throws away). Effective coverage — slots with an alternative AND a negative — is printed for both; the raw 45.6% is not the supervised fraction.


In [ ]:
NEG_VARIANTS = {"clean": ["--strict-lexical", "--antonyms"], "fragments": ["--fragments-only"]}
NEG_FILES = {name: LEAF / f"synonyms_train_negatives_{name}.jsonl" for name in NEG_VARIANTS}
# The grid the gate was pre-registered over on 2026-09-18, before any of it was run.
PREREGISTERED_GRID = [("uniform", 0.25), ("uniform", 0.5),
                      ("within_kl", 0.25), ("within_kl", 0.5), ("within_kl", 1.0)]
# Added 2026-09-21, AFTER that grid was screened and every arm failed on STS alone --
# Llama uniform g0.25 missed the floor by .0004 (the three-seed STS sd is .0005), Qwen
# by .0063.  These two resolve the knee of the frontier between Zhang and g=0.25.  They
# are frontier points, NOT gate candidates: hybrid_gate() scores and prints them but
# refuses to select them, because extending a grid downward after reading a near-miss is
# exactly how a pre-registration gets spent.  The curve was the pre-registered outcome
# when nothing passed; adding points to a curve is resolution, not a second attempt.
POSTHOC_GRID = [("uniform", 0.125), ("uniform", 0.0625)]
HYBRID_GRID = PREREGISTERED_GRID + POSTHOC_GRID
# "within_kl:0.5" -- set ONLY from the gate above, then rerun with RUN_MULTISEED.
SELECTED_HYBRID = os.environ.get("SELECTED_HYBRID")
if RUN_HYBRID:
    for kind, weight in HYBRID_GRID:
        OBJECTIVE_RUNS[f"zhang_plus_{kind}_g{weight}_seed42"] = train_flat(
            "zhang_plus_aux", 42, 1.0, alt_aux=kind, alt_aux_weight=weight)
    # Confirmation of the arm the gate below selected.  RUN_CONFIRM is NOT used
    # for this: that flag also relaunches the alternative-uniform and legacy
    # contrastive arms, which are ablations here, not the method.
    if SELECTED_HYBRID:
        kind, weight = SELECTED_HYBRID.split(":"); weight = float(weight)
        assert (kind, weight) in HYBRID_GRID, "SELECTED_HYBRID must be one of the screened arms"
        for seed in SEEDS:
            OBJECTIVE_RUNS[f"calibrated_seed{seed}"] = train_flat(
                "zhang_plus_aux", seed, 1.0, alt_aux=kind, alt_aux_weight=weight)
        # Same adapter as the seed-42 screen arm; one key per adapter.
        OBJECTIVE_RUNS.pop(f"zhang_plus_{kind}_g{weight}_seed42", None)

if RUN_NEGATIVE_CONTROLS:
    topk_glob = DATA / "c4" / MODEL_TAG / "prompting" / "topk_*.jsonl"
    if not list(topk_glob.parent.glob(topk_glob.name)):
        print("no top-k shards restored; the negative controls are skipped, not faked")
    else:
        assert SELECTED_ALPHA is not None, "the negative controls reuse the locked alpha"
        for name, flags in NEG_VARIANTS.items():
            report = MODEL_OUT / f"contrastive_negative_report_{name}.json"
            if not NEG_FILES[name].is_file():
                run([sys.executable, "data/build_contrastive_negatives.py",
                     "--source", LEAF / "synonyms_train.jsonl", "--topk", topk_glob,
                     "--output", NEG_FILES[name], "--report", report,
                     "--max-cosine", "0.35", "--max-negatives", "20",
                     # Coverage counted the way the TRAINER counts: a multi-token
                     # candidate never reaches the loss, so the raw fraction
                     # overstates what is actually supervised.
                     "--tokenizer", BASE_MODEL, *flags], cwd=EXT)
            if report.is_file():
                stats = json.loads(report.read_text())
                print(f"{name}: raw coverage {stats['negative_coverage']:.3f}, "
                      f"EFFECTIVE contrastive coverage {stats['effective_contrastive_coverage']:.3f}")
            # A distinct method name per file: adapter_path() does not see train_file.
            OBJECTIVE_RUNS[f"contrast_{name}_negatives_seed42"] = train_flat(
                f"alternative_contrastive_{name}", 42, SELECTED_ALPHA, objective="uniform",
                slot_ntp_weight=1.0, exclude_target=True, contrast_beta=1.0,
                train_file=NEG_FILES[name])

if RUN_HYBRID or RUN_NEGATIVE_CONTROLS:
    save_runs(OBJECTIVE_RUNS, "task15b")

## Locked confirmation

Lock $\alpha$ and $\beta$ from the seed-42 screen's decision cell further down; never choose them per seed. This cell sits before evaluation on purpose, so one Run All trains the new seeds and then scores them. Seeds 42, 123, 2024 for the alternative-only uniform arm and, if promoted, the contrastive arm. The seed-42 adapters already exist and resume for free.


In [ ]:
SELECTED_BETA = float(os.environ["SELECTED_BETA"]) if os.environ.get("SELECTED_BETA") else None
if RUN_CONFIRM:
    assert SELECTED_ALPHA is not None, "set SELECTED_ALPHA from the alpha gate first"
    for seed in SEEDS:
        OBJECTIVE_RUNS[f"alternative_uniform_seed{seed}"] = train_flat(
            "alternative_uniform", seed, SELECTED_ALPHA, objective="uniform", slot_ntp_weight=1.0,
            exclude_target=True)
        if SELECTED_BETA is not None:
            OBJECTIVE_RUNS[f"contrastive_seed{seed}"] = train_flat(
                "alternative_contrastive", seed, SELECTED_ALPHA, objective="uniform",
                slot_ntp_weight=1.0, exclude_target=True, contrast_beta=SELECTED_BETA, train_file=NEG_TRAIN)
    # The screen already trained seed 42 at the locked values and adapter_path()
    # maps both calls to one directory; two keys on one adapter would score it twice.
    OBJECTIVE_RUNS.pop(f"alternative_uniform_alpha{SELECTED_ALPHA}_seed42", None)
    if SELECTED_BETA is not None:
        OBJECTIVE_RUNS.pop(f"contrast_alpha{SELECTED_ALPHA}_beta{SELECTED_BETA}_seed42", None)
    for label, path in OBJECTIVE_RUNS.items(): sync_small_artifacts(path, LOG_DIR / "task15b_logs" / label)
    audit_outputs()
save_runs(OBJECTIVE_RUNS, "task15b")

## Evaluation

Every checkpoint of this pass — Task 15's arms and this notebook's — is scored by the same evaluators. A validation pass of the perplexity and concept-set evaluators exists only for choosing $\alpha$; every other number is C4 test, SWORDS dev, the nine STS tasks and bm-semlex.


In [ ]:
if RUN_EVAL and not OBJECTIVE_RUNS:
    print("no screen arms for this model here; the screen evaluation is skipped, "
          "not run on the baselines alone")
if RUN_EVAL and OBJECTIVE_RUNS:
    OBJECTIVE_RUNS = restore_all(OBJECTIVE_RUNS)
    # The question is "does this beat Zhang", so Zhang's arms sit IN this table:
    # pretrained as the reference row, NTP and augmented NTP as matched controls,
    # randomized at both weights as the semantic control, and set-marginal at
    # lambda=1 as the method being improved on.  They come from Task 15's manifest;
    # an arm Task 15 never trained is reported as absent, never retrained here.
    task15_runs = restore_all(load_runs("task15"))
    BASELINE_LABELS = ("ntp_seed42", "augmented_ntp_seed42", "randomized_seed42",
                       "randomized_lambda1.0_seed42", "zhang_seed42", "zhang_lambda1.0_seed42")
    baseline_runs = {label: task15_runs[label] for label in BASELINE_LABELS if label in task15_runs}
    # RUN_CONFIRM in Task 15 pops zhang_lambda1.0_seed42 in favour of zhang_seed42;
    # both name ONE adapter, so keep whichever exists and never both.
    if "zhang_seed42" in baseline_runs:
        baseline_runs.pop("zhang_lambda1.0_seed42", None)
    for label in BASELINE_LABELS[:-1]:
        if label not in baseline_runs and not (label == "zhang_seed42" and "zhang_lambda1.0_seed42" in baseline_runs):
            print("Task 15 never trained this baseline; the table will lack it:", label)
    all_runs = {**baseline_runs, **OBJECTIVE_RUNS}
    checkpoints = [BASE_MODEL, *map(str, all_runs.values())]
    result_dir = SCREEN_DIR
    # Each evaluator is skipped only when its own output already scores every
    # checkpoint of this pass, so a disconnect costs at most one evaluator.
    # VALIDATION pass first: this is the only thing the alpha choice may read.
    guarded(result_dir / "val_perplexity.json", checkpoints,
            [sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_val.jsonl",
             "--output", result_dir / "val_perplexity.json"], EXT, "validation perplexity")
    guarded(result_dir / "val_concept_sets.json", checkpoints,
            [sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_val.jsonl",
             "--output", result_dir / "val_concept_sets.json"], EXT, "validation concept sets")
    guarded(result_dir / "perplexity.json", checkpoints,
            [sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "perplexity.json"], EXT, "perplexity")
    guarded(result_dir / "concept_sets.json", checkpoints,
            [sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "concept_sets.json"], EXT, "concept sets")
    task15_dir = RESULT_DIR
    for label, checkpoint in {"pretrained": BASE_MODEL, **all_runs}.items():
        display_label = label.replace("_", " ")
        csv_output = result_dir / f"sts_{display_label}.csv"
        # STS is deterministic per checkpoint and Task 15 already scored the
        # baselines, so reuse its file rather than spending 3.5 min re-deriving it.
        previous = task15_dir / csv_output.name
        if not csv_output.is_file() and previous.is_file():
            shutil.copy2(previous, csv_output)
        if sts_covered(csv_output):
            print("resume: STS already scored, skipping", display_label)
            continue
        mteb_args = [sys.executable, "eval/eval_mteb.py", "--base-model", BASE_MODEL,
                     "--dataset", "c4", "--dataset-type", "embedding", "--tasks", "sts",
                     "--run-label", display_label, "--csv-output", csv_output,
                     "--mteb-output-root", result_dir / "mteb_raw"]
        mteb_args += ["--no-adapter"] if checkpoint == BASE_MODEL else ["--adapter-path", checkpoint]
        run(mteb_args, cwd=EXT)
    guarded(result_dir / "swords.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_swords.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--swords_json", MAIN / "data/swords/swords-v1.1_dev.json.gz",
             "--results_json", result_dir / "swords.json", "--modes", "left", "full"], MAIN, "SWORDS")
    # One paired interval per reference, looked up by label so an index can never
    # drift onto the wrong arm.  vs zhang is the headline; vs randomized is what
    # says whether a gain is semantic rather than distributional; vs the selected
    # alternative-uniform arm is the only fair test of the contrastive term itself.
    references = {"pretrained": BASE_MODEL, **{label: str(path) for label, path in baseline_runs.items()}}
    # EVERY alternative-only uniform arm is a reference, not just the selected one.
    # Contrastive has to beat the cheaper way of buying the same concept pressure,
    # which is turning alpha up.  A same-alpha comparison alone cannot show that:
    # it credits the negatives with a gain a larger alpha also delivers, which is
    # the identical error this paper accuses set-marginal training of.
    # Includes the confirmation arms (alternative_uniform_seed123, ...), so each
    # contrastive seed has a paired interval against the uniform arm of the SAME
    # seed -- the one comparison that isolates the contrastive term across seeds.
    for key, path in OBJECTIVE_RUNS.items():
        if key.startswith("alternative_uniform"):
            references[key] = str(path)
    for label, reference in references.items():
        run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/paired_benchmark_ci.py",
             "--kind", "swords", "--results-json", result_dir / "swords.json",
             "--baseline-index", checkpoints.index(reference),
             "--output", result_dir / f"swords_paired_ci_vs_{label}.json"], cwd=MAIN)
    guarded(result_dir / "bm_semlex.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_bm_semlex.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--data", MAIN / "data/bm_semlex/curated_200.tsv",
             "--results_json", result_dir / "bm_semlex.json"], MAIN, "bm-semlex")
    manifest = {"pretrained": BASE_MODEL, **{label.replace("_", " "): str(path) for label, path in all_runs.items()}}
    (result_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    run([sys.executable, MAIN / "scripts/summarize_concept_experiments.py",
         "--manifest", result_dir / "manifest.json", "--result-dir", result_dir,
         "--output", result_dir / "flat_extension_table.csv"], cwd=MAIN)

## Decision cell

Reads only what the evaluation cell wrote and prints every gate with its number, so the choice is auditable from the notebook output alone. First pass: the $\alpha$ gate (validation only). After `SELECTED_ALPHA` is set and the $\beta$ arms are scored: the promotion gates and the headline.

Headline rule: contrastive if it passes every promotion gate; otherwise alternative-only uniform if it passes the same gates minus the vs-itself clause; otherwise this is a negative result and is reported as one.


In [ ]:
import csv
GLOBAL_NLL_SLACK, OBSERVED_NLL_SLACK, STS_SLACK, BM_SLACK, MIN_PROB_RATIO = 0.20, 0.10, 0.005, 0.02, 0.5
result_dir = SCREEN_DIR

def _by_ckpt(name):
    path = result_dir / name
    return {str(r["checkpoint"]): r for r in json.loads(path.read_text())} if path.is_file() else {}

def _sts(label):
    path = result_dir / f"sts_{label.replace('_', ' ')}.csv"
    if not path.is_file():
        return None
    with path.open(encoding="utf-8") as handle:
        scores = [float(r["main_score"]) for r in csv.DictReader(handle)]
    return sum(scores) / len(scores) if scores else None

def _paired(reference_label, candidate):
    path = result_dir / f"swords_paired_ci_vs_{reference_label}.json"
    if not path.is_file():
        return {}
    for entry in json.loads(path.read_text()):
        if str(entry["candidate"]) == str(candidate):
            return entry["metrics"]
    return {}

def _sig(metrics, key, better):
    # (candidate - reference, True when the 95% CI excludes zero on the good side)
    value = metrics.get(key)
    if not value or value.get("candidate_minus_baseline") is None:
        return None, None
    lo, hi = value["ci95"]
    return value["candidate_minus_baseline"], (lo > 0) if better == "up" else (hi < 0)

def _fmt(x):
    return "n/a" if x is None else f"{x:.4f}"

manifest_path = result_dir / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
runs = {label.replace(" ", "_"): str(path) for label, path in manifest.items()}
ntp = runs.get("ntp_seed42")
zhang_label = "zhang_seed42" if "zhang_seed42" in runs else "zhang_lambda1.0_seed42"
zhang, randomized = runs.get(zhang_label), runs.get("randomized_seed42")
val_c, val_p = _by_ckpt("val_concept_sets.json"), _by_ckpt("val_perplexity.json")
tst_p, bm = _by_ckpt("perplexity.json"), _by_ckpt("bm_semlex.json")

print("== alpha gate: C4 VALIDATION only ==")
g_ntp = val_p.get(ntp, {}).get("global", {}).get("mean_nll")
# The observed target is referenced against NTP, never against Zhang.  Zhang's
# supervised set CONTAINS the observed token, so its objective actively drives
# that NLL below NTP's; asking an alternative-only arm to match it would be
# asking it to do the one thing it exists not to do.  NTP is the matched control,
# and the question here is only "is the observed target damaged?" -- which
# --slot-ntp-weight 1.0 is what prevents.
o_ntp = val_c.get(ntp, {}).get("observed_target_nll")
mp_zhang = val_c.get(zhang, {}).get("minimum_candidate_probability")
alpha_rows = []
for label, ck in runs.items():
    if not label.startswith("alternative_uniform_alpha"):
        continue
    c, p = val_c.get(ck, {}), val_p.get(ck, {})
    alt, g = c.get("alternative_nll"), p.get("global", {}).get("mean_nll")
    o, mp = c.get("observed_target_nll"), c.get("minimum_candidate_probability")
    gates = {"global<=ntp+.20": None if None in (g, g_ntp) else g <= g_ntp + GLOBAL_NLL_SLACK,
             "observed<=ntp+.10": None if None in (o, o_ntp) else o <= o_ntp + OBSERVED_NLL_SLACK,
             "min_prob>=.5*zhang": None if None in (mp, mp_zhang) else mp >= MIN_PROB_RATIO * mp_zhang}
    alpha_rows.append((label, alt, gates))
    print(f"  {label:38} alt_nll={_fmt(alt)} global={_fmt(g)} observed={_fmt(o)} min_prob={_fmt(mp)}  "
          + "  ".join(f"{k}:{'n/a' if v is None else ('PASS' if v else 'FAIL')}" for k, v in gates.items()))
passing = [r for r in alpha_rows if r[1] is not None and all(r[2].values())]
if passing:
    best = min(passing, key=lambda r: r[1])
    print("  -> SELECTED_ALPHA =", best[0].split("alpha", 1)[1].split("_", 1)[0])
elif alpha_rows:
    print("  -> no alpha passes every gate.  Report that; do not loosen the gates.")
else:
    print("  (no alternative-uniform arms scored yet)")

print("== promotion gates: SWORDS DEV paired intervals, C4 test, STS, bm-semlex ==")
g_ntp_test = tst_p.get(ntp, {}).get("global", {}).get("mean_nll")
sts_zhang = _sts(zhang_label)
acc_zhang = bm.get(zhang, {}).get("left", {}).get("accuracy")

def promotion(label, ck, self_label=None):
    vz, vr = _paired(zhang_label, ck), _paired("randomized_seed42", ck)
    vs = _paired(self_label, ck) if self_label else {}
    g = tst_p.get(ck, {}).get("global", {}).get("mean_nll")
    sts, acc = _sts(label), bm.get(ck, {}).get("left", {}).get("accuracy")
    d_gap_z, s_gap_z = _sig(vz, "gap", "up")
    d_auroc_s, s_auroc_s = _sig(vs, "auroc", "up")
    d_gap_r, _ = _sig(vr, "gap", "up")
    _, s_auroc_r = _sig(vr, "auroc", "up")
    _, s_rms_r = _sig(vr, "rejected_mass_share", "down")
    d_alt_z, s_alt_z = _sig(vz, "alternatives_nll", "down")
    # Frontier gate.  An arm is DOMINATED when some uniform arm already matches or
    # beats its ranking at no greater language-modelling cost -- that arm's gain is
    # the alpha curve, not the negatives.  Uniform arms that cost MORE global NLL
    # are excluded: losing to a costlier arm is a trade, not a domination.
    dominated = []
    for u_label, u_ck in (runs.items() if self_label else ()):
        u_global = tst_p.get(u_ck, {}).get("global", {}).get("mean_nll")
        if not u_label.startswith("alternative_uniform_alpha") or str(u_ck) == str(ck):
            continue
        if None in (u_global, g) or u_global > g:
            continue
        _, beats = _sig(_paired(u_label, ck), "gap", "up")
        if beats is not None:
            dominated.append(not beats)
    on_frontier = None if not dominated else not any(dominated)
    # Two groups, reported and decided separately.  DISCRIMINATION is the claim:
    # does this arm rank human-labelled substitutes better than Zhang, and better
    # than the randomized control Zhang cannot separate from?  RETENTION is the
    # price: are STS and language modelling preserved?  Collapsing them into one
    # verdict turns "wins the claim, pays on STS" -- a reportable trade-off and
    # the most likely real outcome -- into the same output as "nothing worked".
    discrimination = {
        f"GAP > zhang (CI excl 0)  [{_fmt(d_gap_z)}]": None if s_gap_z is None else bool(s_gap_z),
        f"AUROC > same non-contrastive arm (CI excl 0)  [{_fmt(d_auroc_s)}]": (None if not self_label or s_auroc_s is None else bool(s_auroc_s)),
        f"GAP >= randomized lambda.25  [{_fmt(d_gap_r)}]": None if d_gap_r is None else d_gap_r >= 0,
        "AUROC or rejected-mass share better than randomized (CI excl 0)": (None if s_auroc_r is None and s_rms_r is None else bool(s_auroc_r or s_rms_r)),
        f"human-accepted alt NLL < zhang (CI excl 0)  [{_fmt(d_alt_z)}]": None if s_alt_z is None else bool(s_alt_z),
        "GAP > every uniform arm costing no more global NLL (CI excl 0)": on_frontier,
    }
    retention = {
        f"global NLL <= ntp+.20  [{_fmt(g)} vs {_fmt(g_ntp_test)}]": None if None in (g, g_ntp_test) else g <= g_ntp_test + GLOBAL_NLL_SLACK,
        f"STS >= zhang-.005  [{_fmt(sts)} vs {_fmt(sts_zhang)}]": None if None in (sts, sts_zhang) else sts >= sts_zhang - STS_SLACK,
        f"bm-semlex >= zhang-2pp  [{_fmt(acc)} vs {_fmt(acc_zhang)}]": None if None in (acc, acc_zhang) else acc >= acc_zhang - BM_SLACK,
    }
    print(f"  {label}")
    for heading, group in (("discrimination (the claim)", discrimination), ("retention (the price)", retention)):
        print(f"    {heading}")
        for name, ok in group.items():
            print(f"      {'n/a ' if ok is None else ('PASS' if ok else 'FAIL')}  {name}")
    def _all(group):
        known = [ok for ok in group.values() if ok is not None]
        return bool(known) and all(known)
    return _all(discrimination), _all(retention)

winners = {"contrastive": [], "alternative_uniform": []}
retained = {}
for label, ck in runs.items():
    if label.startswith("contrast_alpha"):
        alpha = label.split("alpha", 1)[1].split("_", 1)[0]
        discriminates, retains = promotion(label, ck, self_label=f"alternative_uniform_alpha{alpha}_seed42")
        if discriminates:
            winners["contrastive"].append(label)
        retained[label] = retains
    elif label.startswith("alternative_uniform_alpha"):
        discriminates, retains = promotion(label, ck)
        if discriminates:
            winners["alternative_uniform"].append(label)
        retained[label] = retains

def announce(kind, labels):
    for label in labels:
        price = ("and pays nothing: every retention gate holds" if retained.get(label)
                 else "but FAILS a retention gate above -- that trade-off is a result, report it")
        print(f"  {kind} wins every discrimination gate: {label} {price}")

print("== headline ==")
if winners["contrastive"]:
    announce("contrastive", winners["contrastive"])
    print("  -> beta is TUNED on SWORDS dev (test is never read here), so any passing")
    print("     beta is admissible.  Take the smallest unless the frontier margin is")
    print("     monotone in beta, in which case take the largest that still retains,")
    print("     and record the choice and its reason before the seeds are run.")
elif winners["alternative_uniform"]:
    announce("alternative-only uniform", winners["alternative_uniform"])
    print("  -> contrastive did not separate from it; the uniform arm is the headline candidate")
else:
    print("  no arm wins every discrimination gate: this is a negative result and is reported as one")

## Hybrid gate (automatic)

Applied to the H1/H2 arms only, on SWORDS **dev**, and evaluated before any of these numbers is read by hand. Thresholds are derived from this notebook's own Zhang and NTP rows rather than hardcoded, so the same cell gates a second model without edits.

An arm passes only if it beats Zhang on the axis Zhang provably does not move (human-accepted alternative NLL) **and** on ranking, stays within the retention allowances, and also beats the randomized control. The smallest weight that passes is selected; ties go to the smaller weight. If nothing passes, that is the result and the paper reports the trade-off curve — the cell does not relax a threshold to manufacture a winner.


In [ ]:
STS_ALLOWANCE = 0.005        # vs Zhang
GLOBAL_NLL_ALLOWANCE = 0.20  # vs NTP
OBSERVED_NLL_ALLOWANCE = 0.10

def _gate_rows():
    table = SCREEN_DIR / "flat_extension_table.csv"
    if not table.is_file():
        print("no flat_extension_table.csv yet; run RUN_EVAL first"); return None
    import csv
    return {r["method"]: r for r in csv.DictReader(table.open())}

def _ci(path, run_path, metric):
    # Returns (delta, significant) for one candidate in one paired-CI file.
    if not Path(path).is_file():
        return None
    for entry in json.loads(Path(path).read_text()):
        if str(entry["candidate"]) != str(run_path):
            continue
        if metric not in entry["metrics"]:
            return None
        d = entry["metrics"][metric]
        low, high = d["ci95"]
        return d["candidate_minus_baseline"], (low * high > 0)
    return None

def hybrid_gate(verbose=True):
    rows = _gate_rows()
    if rows is None: return None
    zhang = rows.get("zhang seed42") or rows.get("zhang lambda1.0 seed42")
    ntp = rows.get("ntp seed42")
    if not zhang or not ntp:
        print("Task 15 baselines missing from the table; cannot gate"); return None
    sts_floor = float(zhang["sts_mean"]) - STS_ALLOWANCE
    nll_ceiling = float(ntp["global_nll"]) + GLOBAL_NLL_ALLOWANCE
    obs_ceiling = float(ntp["swords_observed_target_nll"]) + OBSERVED_NLL_ALLOWANCE
    print(f"thresholds -> STS >= {sts_floor:.4f} | global NLL <= {nll_ceiling:.4f} "
          f"| observed-target NLL <= {obs_ceiling:.4f}")

    vs_zhang = SCREEN_DIR / ("swords_paired_ci_vs_zhang_seed42.json"
                             if (SCREEN_DIR / "swords_paired_ci_vs_zhang_seed42.json").is_file()
                             else "swords_paired_ci_vs_zhang_lambda1.0_seed42.json")
    vs_rand = SCREEN_DIR / "swords_paired_ci_vs_randomized_seed42.json"

    passing = []
    for kind, weight in HYBRID_GRID:
        label = f"zhang_plus_{kind}_g{weight}_seed42"
        run_path = OBJECTIVE_RUNS.get(label)
        row = rows.get(label.replace("_", " "))
        if row is None or run_path is None:
            if verbose: print(f"  {label:34} not trained/scored yet")
            continue
        checks = {
            "STS": float(row["sts_mean"]) >= sts_floor,
            "globalNLL": float(row["global_nll"]) <= nll_ceiling,
            "obsNLL": float(row["swords_observed_target_nll"]) <= obs_ceiling,
        }
        for metric, key, want_negative in (("alternatives_nll", "altNLL<Zhang", True),
                                           ("gap", "GAP>Zhang", False),
                                           ("auroc", "AUROC>Zhang", False)):
            got = _ci(vs_zhang, run_path, metric)
            checks[key] = bool(got and got[1] and ((got[0] < 0) == want_negative))
        got = _ci(vs_rand, run_path, "gap")
        checks["GAP>random"] = bool(got and got[1] and got[0] > 0)

        ok = all(checks.values())
        posthoc = (kind, weight) in POSTHOC_GRID
        if verbose:
            failed = [k for k, v in checks.items() if not v]
            print(f"  {label:36} {'PASS' if ok else 'FAIL'}"
                  f"{'  [post-hoc, not selectable]' if posthoc else ''}"
                  f"  STS {float(row['sts_mean']):.4f}"
                  f"  gNLL {float(row['global_nll']):.4f}"
                  f"  altNLL {float(row['swords_alternative_nll']):.3f}"
                  + ("" if ok else f"   failed: {', '.join(failed)}"))
        # A post-hoc point never enters `passing`: it is reported on the frontier curve
        # and cannot become the method by passing a gate it was added after reading.
        if ok and not posthoc: passing.append((weight, kind))

    if not passing:
        print("\nNo hybrid arm passes every gate. That is the result: report the "
              "STS-vs-substitution trade-off curve, do not relax a threshold.")
        return None
    weight, kind = sorted(passing)[0]
    print(f"\nSELECTED_HYBRID = {kind}:{weight}   (smallest passing weight of "
          f"{len(passing)}; set it in the environment and rerun with RUN_MULTISEED)")
    return f"{kind}:{weight}"

GATE_CHOICE = hybrid_gate()

## Verified supervision: one slot objective per arm

The question this stage answers, with everything else held equal:

> Does teaching every valid alternative to outrank verifier-filtered negatives improve contextual substitution over maximising the total probability of a concept set?

Same training file for every arm (`synonyms_train_verified.jsonl`: positives pruned by the verifier's low tail, negatives mined from the model's own top-$k$ pool through the same tail), same schedule and initialisation, NTP everywhere else. What differs is **only** what sits at the concept slot:

| arm | at the slot | positives | negatives |
|---|---|---|---|
| `zhang_seed42` (Task 15) | released set marginal | original | — |
| `verified_zhang` | released set marginal | pruned | — |
| `verified_hybrid` | set marginal $+\gamma\,$uniform | pruned | — |
| `verified_pool` | $-\log\frac{P(A)}{P(A)+P(N)}$ | pruned | yes |
| `verified_rank` | $-\frac1{|A|}\sum_a\log\frac{p_a}{p_a+P(N)}$ | pruned | yes |
| `verified_list_uniform` | $-\frac1{|A|}\sum_a\log\frac{p_a}{P(A)+P(N)}$ | pruned | yes |
| `verified_list_verifier` | $-\sum_a w_a\log\frac{p_a}{P(A)+P(N)},\ w_a\propto\max(s_a-\theta,0)$ | pruned | yes |

The four discriminative arms exclude the observed token from $A$ and keep slot NTP at 1.0 (the trainer refuses any other combination, and refuses `--contrast-beta` or `--alt-aux` on top of them). A slot carries the term only when at least one alternative **and** one negative survive tokenization; otherwise it contributes exactly zero and stays in the denominator.

Two claims, gated separately below and never merged: **contrast helps** (a discriminative arm beats `verified_zhang` on GAP and AUROC with paired intervals excluding zero, inside the STS and observed-target allowances) and **per-positive helps** (`verified_rank` beats `verified_pool` the same way). Beating the old noisy baseline establishes neither.

Adapters live under a method name that carries the training file's SHA-256, and a resume is refused if the recorded hash differs, so a pruned-positive run can never reuse an older adapter.


In [ ]:
VERIFIED_DIR = MODEL_OUT / "task15b_verified"
VERIFIED_DIR.mkdir(parents=True, exist_ok=True)
VERIFIED_TRAIN = Path(os.environ.get("VERIFIED_TRAIN_FILE") or (LEAF / "synonyms_train_verified.jsonl"))
VERIFIED_LAMBDA = float(os.environ.get("VERIFIED_LAMBDA", "1.0"))
VERIFIED_GAMMA = float(os.environ.get("VERIFIED_GAMMA") or (0.125 if MODEL_TAG == "llama-3.2-1b" else 0.0625))
VERIFIED_RUNS = load_runs("task15b_verified")
DISCRIMINATIVE = ("pool", "rank", "list_uniform", "list_verifier")

if RUN_VERIFIED:
    assert VERIFIED_TRAIN.is_file(), f"{VERIFIED_TRAIN} missing: run scripts/build_verified_negatives.py first"
    DATA_HASH = sha256(VERIFIED_TRAIN)
    DATA_TAG = f"data_{DATA_HASH[:8]}"
    print("verified train file:", VERIFIED_TRAIN, "| sha256", DATA_HASH[:16] + "...",
          "| lambda", VERIFIED_LAMBDA, "| gamma", VERIFIED_GAMMA)

    ARMS = {
        "verified_zhang":         dict(objective="set_marginal", weight=1.0),
        "verified_hybrid":        dict(objective="set_marginal", weight=1.0,
                                       alt_aux="uniform", alt_aux_weight=VERIFIED_GAMMA),
        "verified_pool":          dict(objective="pool"),
        "verified_rank":          dict(objective="rank"),
        "verified_list_uniform":  dict(objective="list_uniform"),
        "verified_list_verifier": dict(objective="list_verifier"),
    }

    def launch(label, spec, *, prefix="", epochs=5, max_samples=None):
        objective = spec["objective"]
        weight = spec.get("weight", VERIFIED_LAMBDA)
        alt_aux, alt_w = spec.get("alt_aux", "none"), spec.get("alt_aux_weight", 0.0)
        kw = dict(objective=objective, train_file=VERIFIED_TRAIN, epochs=epochs,
                  max_samples=max_samples, alt_aux=alt_aux, alt_aux_weight=alt_w)
        if objective in DISCRIMINATIVE:
            kw.update(exclude_target=True, slot_ntp_weight=1.0)
        # The data hash is part of the METHOD name, so this path can only ever hold
        # an adapter trained on this exact file -- and the recorded hash is checked
        # anyway before a resume is accepted.
        method = f"{prefix}{label}_{DATA_TAG}"
        aux_tag = "" if alt_aux == "none" else f"_aux_{alt_aux}_{alt_w}"
        expected = adapter_path(method, 42, f"lambda_{weight}_beta_0.0{aux_tag}")
        if RESUME_FINISHED_RUNS and finished(expected):
            recorded = json.loads((expected / "concept_training_config.json").read_text()) \
                if (expected / "concept_training_config.json").is_file() else {}
            if recorded.get("train_file_sha256") != DATA_HASH:
                raise RuntimeError(f"{expected} was trained on a different file "
                                   f"({recorded.get('train_file_sha256')}); refusing to resume it")
        return train_flat(method, 42, weight, **kw)

    if VERIFIED_SMOKE_STEPS:
        print(f"== smoke: ~{VERIFIED_SMOKE_STEPS} steps per arm, first logged magnitudes ==")
        for label in ("verified_zhang", "verified_pool", "verified_rank",
                      "verified_list_uniform", "verified_list_verifier"):
            out = launch(label, ARMS[label], prefix="smoke_", epochs=1,
                         max_samples=VERIFIED_SMOKE_STEPS * 16)
            history = out / "training_history.jsonl"
            rows = [json.loads(l) for l in history.read_text().splitlines() if l.strip()] \
                if history.is_file() else []
            first = next((r for r in rows if "concept_loss" in r), None)
            print(f"  {label:24s}", {k: round(first[k], 4) for k in
                  ("ce_loss", "concept_loss", "concept_eligible_share") if k in first}
                  if first else "no logged step (raise VERIFIED_SMOKE_STEPS above logging_steps)")
        print("Rule, declared before any result: VERIFIED_LAMBDA = 1.0 unless an arm's first "
              "concept_loss is more than 3x or less than 1/3 of verified_zhang's, in which case "
              "the nearest power of two.  Set it in the environment, set VERIFIED_SMOKE_STEPS = 0, rerun.")
    else:
        for label, spec in ARMS.items():
            VERIFIED_RUNS[label] = launch(label, spec)
        for label, path in VERIFIED_RUNS.items():
            sync_small_artifacts(path, LOG_DIR / "task15b_verified_logs" / label)
        save_runs(VERIFIED_RUNS, "task15b_verified")

if RUN_VERIFIED and RUN_EVAL and not VERIFIED_SMOKE_STEPS:
    # Its OWN result directory: the baselines plus these six, about ten checkpoints,
    # instead of re-scoring the whole screen every time an arm is added.
    VERIFIED_RUNS = restore_all(VERIFIED_RUNS)
    task15_runs = restore_all(load_runs("task15"))
    baseline_runs = {label: task15_runs[label] for label in
                     ("ntp_seed42", "randomized_seed42", "zhang_seed42", "zhang_lambda1.0_seed42")
                     if label in task15_runs}
    if "zhang_seed42" in baseline_runs:
        baseline_runs.pop("zhang_lambda1.0_seed42", None)
    all_runs = {**baseline_runs, **VERIFIED_RUNS}
    checkpoints = [BASE_MODEL, *map(str, all_runs.values())]
    result_dir = VERIFIED_DIR
    guarded(result_dir / "val_perplexity.json", checkpoints,
            [sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_val.jsonl",
             "--output", result_dir / "val_perplexity.json"], EXT, "validation perplexity")
    guarded(result_dir / "val_concept_sets.json", checkpoints,
            [sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_val.jsonl",
             "--output", result_dir / "val_concept_sets.json"], EXT, "validation concept sets")
    guarded(result_dir / "perplexity.json", checkpoints,
            [sys.executable, "eval/eval_perplexity_explicit.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "perplexity.json"], EXT, "perplexity")
    guarded(result_dir / "concept_sets.json", checkpoints,
            [sys.executable, "eval/eval_concept_sets.py", "--checkpoints", *checkpoints,
             "--base-model", BASE_MODEL, "--test-jsonl", LEAF / "synonyms_test.jsonl",
             "--output", result_dir / "concept_sets.json"], EXT, "concept sets")
    task15_dir = RESULT_DIR
    for label, checkpoint in {"pretrained": BASE_MODEL, **all_runs}.items():
        display_label = label.replace("_", " ")
        csv_output = result_dir / f"sts_{display_label}.csv"
        # STS is deterministic per checkpoint: reuse the baselines' files from Task 15
        # or the screen rather than re-deriving them.
        for previous in (task15_dir / csv_output.name, SCREEN_DIR / csv_output.name):
            if not csv_output.is_file() and previous.is_file():
                shutil.copy2(previous, csv_output)
        if sts_covered(csv_output):
            print("resume: STS already scored, skipping", display_label)
            continue
        mteb_args = [sys.executable, "eval/eval_mteb.py", "--base-model", BASE_MODEL,
                     "--dataset", "c4", "--dataset-type", "embedding", "--tasks", "sts",
                     "--run-label", display_label, "--csv-output", csv_output,
                     "--mteb-output-root", result_dir / "mteb_raw"]
        mteb_args += ["--no-adapter"] if checkpoint == BASE_MODEL else ["--adapter-path", checkpoint]
        run(mteb_args, cwd=EXT)
    guarded(result_dir / "swords.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_swords.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--swords_json", MAIN / "data/swords/swords-v1.1_dev.json.gz",
             "--results_json", result_dir / "swords.json", "--modes", "left", "full"], MAIN, "SWORDS")
    # Paired intervals against every reference a claim below needs: the released
    # baseline, its pruned-data twin, the continuity arm, and pooled contrast.
    references = {"pretrained": BASE_MODEL, **{label: str(path) for label, path in baseline_runs.items()}}
    for key in ("verified_zhang", "verified_hybrid", "verified_pool"):
        if key in VERIFIED_RUNS:
            references[key] = str(VERIFIED_RUNS[key])
    for label, reference in references.items():
        run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/paired_benchmark_ci.py",
             "--kind", "swords", "--results-json", result_dir / "swords.json",
             "--baseline-index", checkpoints.index(reference),
             "--output", result_dir / f"swords_paired_ci_vs_{label}.json"], cwd=MAIN)
    guarded(result_dir / "bm_semlex.json", checkpoints,
            [sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_bm_semlex.py",
             "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL, "--base_model", BASE_MODEL,
             "--data", MAIN / "data/bm_semlex/curated_200.tsv",
             "--results_json", result_dir / "bm_semlex.json"], MAIN, "bm-semlex")
    manifest = {"pretrained": BASE_MODEL, **{label.replace("_", " "): str(path) for label, path in all_runs.items()}}
    (result_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    run([sys.executable, MAIN / "scripts/summarize_concept_experiments.py",
         "--manifest", result_dir / "manifest.json", "--result-dir", result_dir,
         "--output", result_dir / "flat_extension_table.csv"], cwd=MAIN)

## Verified gate (automatic, two claims, no selection)

Reads only what the cell above wrote. Thresholds come from this table's own rows: STS within .005 of the arm being compared against, observed-target NLL within .10 of NTP, global NLL within .20 of NTP. It prints every check and its number; it selects nothing.


In [ ]:
def verified_gate():
    table = VERIFIED_DIR / "flat_extension_table.csv"
    if not table.is_file():
        print("no verified table yet; run RUN_VERIFIED with RUN_EVAL first"); return
    import csv
    rows = {r["method"]: r for r in csv.DictReader(table.open())}
    ntp = rows.get("ntp seed42")
    if ntp is None:
        print("NTP row missing from the verified table; cannot gate"); return
    nll_ceiling = float(ntp["global_nll"]) + 0.20
    obs_ceiling = float(ntp["swords_observed_target_nll"]) + 0.10
    print(f"retention ceilings -> global NLL <= {nll_ceiling:.4f} | observed-target NLL <= {obs_ceiling:.4f}")

    def beats(candidate, reference_label, sts_reference):
        # (all_pass, checks) for candidate vs one reference, paired on SWORDS dev.
        path = VERIFIED_DIR / f"swords_paired_ci_vs_{reference_label}.json"
        run_path = VERIFIED_RUNS.get(candidate)
        row = rows.get(candidate.replace("_", " "))
        if row is None or run_path is None or not path.is_file():
            return None, {}
        checks = {}
        for metric, key, want_negative in (("gap", "GAP", False), ("auroc", "AUROC", False),
                                           ("alternatives_nll", "acceptedNLL", True)):
            got = _ci(path, run_path, metric)
            checks[key] = bool(got and got[1] and ((got[0] < 0) == want_negative))
        checks["STS"] = float(row["sts_mean"]) >= float(sts_reference["sts_mean"]) - 0.005
        checks["globalNLL"] = float(row["global_nll"]) <= nll_ceiling
        checks["obsNLL"] = float(row["swords_observed_target_nll"]) <= obs_ceiling
        return all(checks.values()), checks

    def show(title, candidate, reference_label):
        ref_row = rows.get(reference_label.replace("_", " "))
        if ref_row is None:
            print(f"  {title:44s} reference row missing"); return
        ok, checks = beats(candidate, reference_label, ref_row)
        if ok is None:
            print(f"  {title:44s} not scored yet"); return
        failed = [k for k, v in checks.items() if not v]
        print(f"  {title:44s} {'PASS' if ok else 'FAIL'}" + ("" if ok else f"   failed: {', '.join(failed)}"))

    print("\n== data effect: pruned positives alone ==")
    zhang_label = "zhang_seed42" if "zhang_seed42" in rows or "zhang seed42" in rows else "zhang_lambda1.0_seed42"
    show("verified_zhang vs zhang (original data)", "verified_zhang", zhang_label)
    print("\n== claim 1: contrast helps (vs verified_zhang, same data) ==")
    for arm in ("verified_pool", "verified_rank", "verified_list_uniform", "verified_list_verifier"):
        show(f"{arm} vs verified_zhang", arm, "verified_zhang")
    print("\n== claim 1, against the released baseline on original data ==")
    for arm in ("verified_pool", "verified_rank", "verified_list_uniform", "verified_list_verifier"):
        show(f"{arm} vs zhang", arm, zhang_label)
    print("\n== claim 2: per-positive ranking helps (vs verified_pool) ==")
    show("verified_rank vs verified_pool", "verified_rank", "verified_pool")
    print("\n== decompositions ==")
    show("verified_list_uniform vs verified_pool  (within-positive calibration)", "verified_list_uniform", "verified_pool")
    show("verified_hybrid vs verified_zhang        (continuity arm on clean data)", "verified_hybrid", "verified_zhang")
    print("\nNothing is selected here.  Confirm across seeds only what passes its own claim; "
          "SWORDS test stays locked until then.")

verified_gate()

## Reporting

Main table: NTP, augmented NTP, Zhang set-marginal, alternative-only uniform, contrastive (if promoted). Control table: pretrained, randomized $\lambda=.25$, randomized $\lambda=1$ (matched weight), target-inclusive uniform ablation. Paired bootstrap intervals on every SWORDS comparison; mean ± sd over three seeds everywhere else. Do not expand to 3B from this notebook until the three-seed 1B result is in; the hierarchy experiment stays deferred.


## SWORDS test — one locked invocation

Everything above is SWORDS **dev**: $\alpha$, $\beta$ and $\gamma$ were all chosen on it, so it is development data and cannot support the headline number. This cell scores test **once**, over every locked arm in a single call, so the method and its baselines are measured on identical rows with identical code.

It refuses to run until `SELECTED_HYBRID` is set, and it writes `swords_test_locked.json` recording the arms and the commit. If that file already exists the cell stops: a second test pass with a changed method is the one thing this protocol exists to prevent. Nothing here may be re-run after reading the result.


In [ ]:
RUN_SWORDS_TEST = False   # set True exactly once, after the method is locked
if RUN_SWORDS_TEST:
    assert SELECTED_HYBRID, "lock the method first: the gate sets SELECTED_HYBRID"
    marker = SCREEN_DIR / "swords_test_locked.json"
    assert not marker.is_file(), (
        f"SWORDS test has already been run: {marker}. Re-running after seeing the "
        "result invalidates it. Delete the marker ONLY if the previous run crashed.")

    # One invocation, every locked arm, in a fixed order.  Baselines come from
    # Task 15's manifest so the test table cannot quietly use a different NTP
    # than the dev table did.
    task15_runs = restore_all(load_runs("task15"))
    locked = {"pretrained": BASE_MODEL}
    for family in ("ntp", "augmented_ntp", "zhang", "randomized"):
        for seed in (42, 123, 2024):
            key = f"{family}_seed{seed}"
            if key in task15_runs:
                locked[key] = str(task15_runs[key])
            else:
                print("MISSING baseline, test table will be incomplete:", key)
    OBJECTIVE_RUNS = restore_all(OBJECTIVE_RUNS)
    for seed in (42, 123, 2024):
        key = f"calibrated_seed{seed}"
        if key in OBJECTIVE_RUNS:
            locked[key] = str(OBJECTIVE_RUNS[key])
        else:
            print("MISSING method seed:", key)
    # The two ablations the paper reports beside the method.
    for key in ("alternative_uniform_seed42", "inclusive_uniform_alpha0.5_seed42"):
        if key in OBJECTIVE_RUNS:
            locked[key] = str(OBJECTIVE_RUNS[key])

    checkpoints = list(locked.values())
    print(f"scoring {len(checkpoints)} locked checkpoints on SWORDS TEST")
    run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/eval_swords.py",
         "--checkpoints", *checkpoints, "--tokenizer_path", BASE_MODEL,
         "--base_model", BASE_MODEL,
         "--swords_json", MAIN / "data/swords/swords-v1.1_test.json.gz",
         "--results_json", SCREEN_DIR / "swords_test.json",
         "--modes", "left", "full"], cwd=MAIN)

    # Seed-matched paired intervals: each method seed against the SAME seed of
    # each baseline.  Averaging over mismatched seeds would fold seed variance
    # into the effect.
    for family in ("zhang", "ntp", "augmented_ntp"):
        for seed in (42, 123, 2024):
            key = f"{family}_seed{seed}"
            if key not in locked:
                continue
            run([sys.executable, MAIN / "transformers/examples/pytorch/language-modeling/paired_benchmark_ci.py",
                 "--kind", "swords", "--results-json", SCREEN_DIR / "swords_test.json",
                 "--baseline-index", checkpoints.index(locked[key]),
                 "--output", SCREEN_DIR / f"swords_test_paired_ci_vs_{key}.json"], cwd=MAIN)

    marker.write_text(json.dumps({
        "selected_hybrid": SELECTED_HYBRID,
        "arms": locked,
        "upstream_commit": UPSTREAM_COMMIT,
        "written": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    }, indent=2))
    print("locked:", marker)

### Reporting the test table

For each metric report (a) mean ± sd over the three seeds of each arm, (b) the seed-matched difference method$_s$ − baseline$_s$ for $s \in \{42,123,2024\}$, and (c) a paired bootstrap over per-target scores, averaged across seeds — not a bootstrap over the seed means, which has three points and no power. A claim needs all three seeds to agree in sign with intervals excluding zero.
